# PatchCamelyon (PCam) — Dataset Investigation

Comprehensive exploration: file inventory, sizes, label distribution, metadata, and sample images.

## 1. Setup and paths

In [ ]:
import os
import sys

# Project root = parent of 'notebooks' if we're in notebooks, else cwd
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'pcam-master'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = os.path.join(PROJECT_ROOT, 'pcam_data')
if not os.path.isdir(DATA_DIR) and os.path.isdir(os.path.join(os.getcwd(), 'pcam_data')):
    DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), 'pcam_data'))
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Exists:", os.path.isdir(DATA_DIR))

## 2. File inventory — names and sizes

In [ ]:
def get_file_inventory(root_dir):
    """List all files under root_dir with sizes."""
    rows = []
    for dirpath, _, filenames in os.walk(root_dir):
        for f in filenames:
            path = os.path.join(dirpath, f)
            try:
                size = os.path.getsize(path)
            except OSError:
                size = None
            rel = os.path.relpath(path, root_dir)
            rows.append((rel, size))
    return rows

inv = get_file_inventory(DATA_DIR)
df_files = pd.DataFrame(inv, columns=['path', 'size_bytes'])
df_files['size_MB'] = (df_files['size_bytes'] / (1024**2)).round(2)
df_files['size_GB'] = (df_files['size_bytes'] / (1024**3)).round(2)
df_files = df_files.sort_values('path')

print("File inventory (all files under pcam_data):")
display(df_files)
print("\nTotal size:", f"{df_files['size_bytes'].sum() / (1024**3):.2f} GB")

## 3. Load dataset and get dimensions

In [ ]:
from keras_pcam.dataset.pcam import load_data

(train_x, train_y, meta_train), (valid_x, valid_y, meta_valid), (test_x, test_y, meta_test) = load_data(data_dir=DATA_DIR)

def summarize_split(name, x, y):
    n = len(x)
    sample = x[0]
    arr = np.array(sample)
    return {"split": name, "n_samples": n, "image_shape": arr.shape}

splits = [
    summarize_split("train", train_x, train_y),
    summarize_split("valid", valid_x, valid_y),
    summarize_split("test", test_x, test_y),
]
df_dims = pd.DataFrame(splits)
print("Dataset dimensions:")
display(df_dims)
print("\nImage shape (H, W, C):", splits[0]["image_shape"])

## 4. Label counts (class distribution)

In [ ]:
def label_counts(y, name):
    flat = np.array(y).flatten()
    uniq, cnt = np.unique(flat, return_counts=True)
    return pd.DataFrame({"split": name, "label": uniq, "count": cnt})

counts = pd.concat([
    label_counts(train_y, "train"),
    label_counts(valid_y, "valid"),
    label_counts(test_y, "test"),
], ignore_index=True)

print("Label distribution (0 = negative, 1 = positive / metastatic):")
display(counts)

pivot = counts.pivot(index="split", columns="label", values="count")
pivot.columns = [f"class_{c}" for c in pivot.columns]
pivot["total"] = pivot.sum(axis=1)
pivot["pct_positive"] = (pivot.get("class_1", 0) / pivot["total"] * 100).round(1)
print("\nPer-split summary:")
display(pivot)

## 5. Metadata tables (columns, dtypes, sample)

In [ ]:
print("Train meta — columns and dtypes:")
print(meta_train.dtypes)
print("\nTrain meta — first 5 rows:")
display(meta_train.head())
print("\nTrain meta — shape:", meta_train.shape)
print("\nTrain meta — describe:")
display(meta_train.describe(include='all'))

In [ ]:
print("Valid meta — head:")
display(meta_valid.head())
print("Test meta — head:")
display(meta_test.head())

## 6. Sample images (positive vs negative)

In [ ]:
def show_samples(x, y, n_per_class=4, title="Samples"):
    y_flat = np.array(y).flatten()
    idx_0 = np.where(y_flat == 0)[0]
    idx_1 = np.where(y_flat == 1)[0]
    np.random.seed(42)
    pick_0 = np.random.choice(idx_0, size=min(n_per_class, len(idx_0)), replace=False)
    pick_1 = np.random.choice(idx_1, size=min(n_per_class, len(idx_1)), replace=False)
    fig, axes = plt.subplots(2, n_per_class, figsize=(2 * n_per_class, 4))
    for i, (ax_row, picks, label) in enumerate(zip(axes, [pick_0, pick_1], ["Negative (0)", "Positive (1)"])):
        for j, idx in enumerate(picks):
            img = np.asarray(x[idx])
            if img.max() > 1:
                img = img / 255.0
            ax_row[j].imshow(img)
            ax_row[j].set_axis_off()
        ax_row[0].set_ylabel(label, fontsize=10)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_samples(train_x, train_y, n_per_class=4, title="Training set samples")

In [ ]:
show_samples(valid_x, valid_y, n_per_class=4, title="Validation set samples")
show_samples(test_x, test_y, n_per_class=4, title="Test set samples")

## 7. Summary

In [ ]:
print("=== PCam dataset summary ===")
print("Data root:", DATA_DIR)
print("Splits: train / valid / test")
print("Train:", len(train_x), "samples")
print("Valid:", len(valid_x), "samples")
print("Test:", len(test_x), "samples")
print("Image shape:", np.array(train_x[0]).shape)
print("Labels: binary (0 = no metastasis, 1 = metastasis in center 32x32)")
print("Meta: slide/origin info per patch (see meta_* DataFrames above).")